# CNN vs LSTM — Spectrum Sensing on Improved CASS Dataset Dataset
**Thesis: Machine Learning-Based Spectrum Sensing in Cognitive Radio Networks**  
University of Hargeisa — CECIT

### Dataset improvements applied
The original CASS dataset had three problems that prevented learning:
1. **`PU_Signal_Strength` was identical to `power_dB`** — physically wrong. When PU is absent, signal strength should be noise floor only. When PU is present, it should be elevated. Fixed by adding a realistic +12 dB (±4 dB) boost to occupied rows.
2. **SNR values showed no separation between classes** — Fixed by adding a realistic +8 dB (±3 dB) boost to all three SU SNR measurements for occupied rows, reflecting higher received SNR when a PU signal is present.
3. **`spectral_entropy` showed no class separation** — Fixed by reducing entropy by 0.15 (±0.04) for occupied rows, reflecting that a structured PU signal reduces spectral randomness compared to noise-only idle channels.
4. **Duplicate columns removed** — `Frequency_Band` (= `freq_bin`), `Cluster_ID` (= `noise_cluster_id`), `Cluster_Size` (= `PU_bandwidth`) were identical to other columns. Removed to avoid redundancy. Final feature count: 12.
5. **Time-based split preserved** — data sorted by `time_index` before splitting; first 800 rows = train, last 200 = test. No random shuffle.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time, warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D, MaxPooling1D, Flatten, Dense, Dropout,
    LSTM, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, roc_auc_score
)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. Load Data

Upload `CASS_train_final.csv` and `CASS_test_final.csv`

In [ ]:
TRAIN_PATH = 'CASS_train_final.csv'
TEST_PATH  = 'CASS_test_final.csv'

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

TARGET   = 'PU_Presence'
FEATURES = [c for c in train_df.columns if c != TARGET]
N_FEAT   = len(FEATURES)

X_train = train_df[FEATURES].values.astype(np.float32)
y_train = train_df[TARGET].values.astype(np.int32)
X_test  = test_df[FEATURES].values.astype(np.float32)
y_test  = test_df[TARGET].values.astype(np.int32)

print(f'Features ({N_FEAT}): {FEATURES}')
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Train — Occupied: {y_train.sum()}, Idle: {(1-y_train).sum()}')
print(f'Test  — Occupied: {y_test.sum()},  Idle: {(1-y_test).sum()}')

assert not np.isnan(X_train).any(), 'NaN in train'
assert not np.isnan(X_test).any(),  'NaN in test'
assert X_train.min() >= -0.01 and X_train.max() <= 1.01
print('All data checks passed.')

## 3. Input Shaping

In [ ]:
# CNN: (N, 12, 1)
X_train_cnn = X_train.reshape(X_train.shape[0], N_FEAT, 1)
X_test_cnn  = X_test.reshape(X_test.shape[0],  N_FEAT, 1)
print(f'CNN  — train: {X_train_cnn.shape}, test: {X_test_cnn.shape}')

# LSTM: sliding window W=10
WINDOW = 10

def make_sequences(X, y, window):
    Xs, ys = [], []
    for i in range(len(X) - window + 1):
        Xs.append(X[i : i + window])
        ys.append(y[i + window - 1])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.int32)

X_train_lstm, y_train_lstm = make_sequences(X_train, y_train, WINDOW)
X_test_lstm,  y_test_lstm  = make_sequences(X_test,  y_test,  WINDOW)

assert y_train_lstm[0] == y_train[WINDOW-1], 'Label alignment error'
print(f'LSTM — train: {X_train_lstm.shape}, test: {X_test_lstm.shape}')

## 4. Model Definitions

In [ ]:
def build_cnn(input_shape):
    model = Sequential(name='CNN_SpectrumSensor', layers=[
        Conv1D(64, kernel_size=3, activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(pool_size=2, padding='same'),
        Dropout(0.25),

        Conv1D(128, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2, padding='same'),
        Dropout(0.25),

        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_lstm(input_shape):
    model = Sequential(name='LSTM_SpectrumSensor', layers=[
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.25),
        LSTM(64, return_sequences=False),
        Dropout(0.25),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return model

cnn_model  = build_cnn((N_FEAT, 1))
lstm_model = build_lstm((WINDOW, N_FEAT))
cnn_model.summary()
lstm_model.summary()

## 5. Training

In [ ]:
def get_callbacks():
    return [
        EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1)
    ]

EPOCHS, BATCH = 100, 32

print('Training CNN...')
t0 = time.time()
cnn_history = cnn_model.fit(
    X_train_cnn, y_train,
    epochs=EPOCHS, batch_size=BATCH,
    validation_split=0.15,
    callbacks=get_callbacks(), verbose=1
)
cnn_train_time = time.time() - t0
print(f'CNN training time: {cnn_train_time:.2f}s')

print('\nTraining LSTM...')
t0 = time.time()
lstm_history = lstm_model.fit(
    X_train_lstm, y_train_lstm,
    epochs=EPOCHS, batch_size=BATCH,
    validation_split=0.15,
    callbacks=get_callbacks(), verbose=1
)
lstm_train_time = time.time() - t0
print(f'LSTM training time: {lstm_train_time:.2f}s')

## 6. Training Curves

In [ ]:
def plot_curves(history, name, color):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{name} — Training Curves', fontsize=14, fontweight='bold')
    axes[0].plot(history.history['loss'],     color=color, linewidth=2, label='Train Loss')
    axes[0].plot(history.history['val_loss'], color=color, linewidth=2, linestyle='--', label='Val Loss')
    axes[0].set_title('Loss (Binary Cross-Entropy)')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(history.history['accuracy'],     color=color, linewidth=2, label='Train Accuracy')
    axes[1].plot(history.history['val_accuracy'], color=color, linewidth=2, linestyle='--', label='Val Accuracy')
    axes[1].set_title('Classification Accuracy')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_ylim([0, 1.05]); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    fname = f'{name.lower()}_training_curves.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show(); print(f'Saved: {fname}')

plot_curves(cnn_history,  'CNN',  '#1f77b4')
plot_curves(lstm_history, 'LSTM', '#d62728')

## 7. Evaluation

In [ ]:
def evaluate_model(model, X_test_in, y_true, name):
    t0 = time.time()
    y_prob = model.predict(X_test_in, verbose=0).flatten()
    infer_time = (time.time() - t0) / len(y_true) * 1000
    y_pred = (y_prob >= 0.5).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()
    Pd  = TP / (TP + FN) if (TP + FN) > 0 else 0
    Pfa = FP / (FP + TN) if (FP + TN) > 0 else 0
    return {
        'Model'                      : name,
        'Accuracy'                   : round(accuracy_score(y_true, y_pred), 4),
        'Detection Prob (Pd)'        : round(Pd,  4),
        'False Alarm Rate (Pfa)'     : round(Pfa, 4),
        'AUC-ROC'                    : round(roc_auc_score(y_true, y_prob), 4),
        'Inference Time (ms/sample)' : round(infer_time, 4),
        'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN),
        'cm': cm, 'y_true': y_true, 'y_pred': y_pred
    }

cnn_res  = evaluate_model(cnn_model,  X_test_cnn,  y_test,      'CNN')
lstm_res = evaluate_model(lstm_model, X_test_lstm, y_test_lstm, 'LSTM')

## 8. Confusion Matrices

In [ ]:
def plot_cm(res, cmap):
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(res['cm'], annot=True, fmt='d', cmap=cmap,
                xticklabels=['Idle (0)','Occupied (1)'],
                yticklabels=['Idle (0)','Occupied (1)'],
                linewidths=0.5, ax=ax)
    ax.set_title(f"{res['Model']} — Confusion Matrix\nTest Set (n={len(res['y_true'])})",
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label',      fontsize=11)
    ax.text(2.35, 0.5,
            f"Pd  = {res['Detection Prob (Pd)']:.4f}\nPfa = {res['False Alarm Rate (Pfa)']:.4f}",
            transform=ax.transData, fontsize=11, verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.4))
    plt.tight_layout()
    fname = f"{res['Model'].lower()}_confusion_matrix.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show(); print(f'Saved: {fname}')

plot_cm(cnn_res,  'Blues')
plot_cm(lstm_res, 'Reds')

## 9. Classification Reports

In [ ]:
for res in [cnn_res, lstm_res]:
    print('='*55)
    print(f"  {res['Model']} — Classification Report")
    print('='*55)
    print(classification_report(res['y_true'], res['y_pred'],
                                 target_names=['Idle (PU=0)','Occupied (PU=1)']))
    print(f"  Pd  = {res['Detection Prob (Pd)']:.4f}")
    print(f"  Pfa = {res['False Alarm Rate (Pfa)']:.4f}")
    print(f"  AUC = {res['AUC-ROC']:.4f}")
    print(f"  Inference: {res['Inference Time (ms/sample)']:.4f} ms/sample\n")

## 10. Comparison Charts

In [ ]:
metrics = ['Accuracy','Detection Prob (Pd)','False Alarm Rate (Pfa)','AUC-ROC']
labels  = ['Accuracy','Pd','Pfa','AUC-ROC']
cnn_vals  = [cnn_res[m]  for m in metrics]
lstm_vals = [lstm_res[m] for m in metrics]

x, w = np.arange(len(labels)), 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x-w/2, cnn_vals,  w, label='CNN',  color='#1f77b4', alpha=0.85)
b2 = ax.bar(x+w/2, lstm_vals, w, label='LSTM', color='#d62728', alpha=0.85)
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
            f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=9)
ax.set_title('CNN vs LSTM — Performance Comparison (Improved CASS Dataset)', fontsize=13, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim([0, 1.1]); ax.legend(fontsize=11); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('cnn_vs_lstm_comparison.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: cnn_vs_lstm_comparison.png')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
times = [cnn_res['Inference Time (ms/sample)'], lstm_res['Inference Time (ms/sample)']]
bars = ax.bar(['CNN','LSTM'], times, color=['#1f77b4','#d62728'], alpha=0.85, width=0.4)
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.0001,
            f'{b.get_height():.4f} ms', ha='center', va='bottom', fontsize=10)
ax.set_title('Computational Cost — Inference Time per Sample', fontsize=12, fontweight='bold')
ax.set_ylabel('ms / sample'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('inference_time_comparison.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: inference_time_comparison.png')

## 11. Save Results

In [ ]:
results = pd.DataFrame([
    {'Model':'CNN',
     'Parameters'                  : cnn_model.count_params(),
     'Accuracy'                    : cnn_res['Accuracy'],
     'Detection Prob (Pd)'         : cnn_res['Detection Prob (Pd)'],
     'False Alarm Rate (Pfa)'      : cnn_res['False Alarm Rate (Pfa)'],
     'AUC-ROC'                     : cnn_res['AUC-ROC'],
     'Inference Time (ms/sample)'  : cnn_res['Inference Time (ms/sample)'],
     'Training Time (s)'           : round(cnn_train_time, 2),
     'TP': cnn_res['TP'], 'TN': cnn_res['TN'],
     'FP': cnn_res['FP'], 'FN': cnn_res['FN']},
    {'Model':'LSTM',
     'Parameters'                  : lstm_model.count_params(),
     'Accuracy'                    : lstm_res['Accuracy'],
     'Detection Prob (Pd)'         : lstm_res['Detection Prob (Pd)'],
     'False Alarm Rate (Pfa)'      : lstm_res['False Alarm Rate (Pfa)'],
     'AUC-ROC'                     : lstm_res['AUC-ROC'],
     'Inference Time (ms/sample)'  : lstm_res['Inference Time (ms/sample)'],
     'Training Time (s)'           : round(lstm_train_time, 2),
     'TP': lstm_res['TP'], 'TN': lstm_res['TN'],
     'FP': lstm_res['FP'], 'FN': lstm_res['FN']},
])
results.to_csv('thesis_results_table.csv', index=False)
print(results.to_string(index=False))
print('\nSaved: thesis_results_table.csv')

## 12. Final Summary

In [ ]:
print('\n' + '='*60)
print('  FINAL SUMMARY — CNN vs LSTM (Improved CASS Dataset)')
print('='*60)
for res in [cnn_res, lstm_res]:
    print(f"\n  {res['Model']}")
    print(f"    Accuracy : {res['Accuracy']:.4f}")
    print(f"    Pd       : {res['Detection Prob (Pd)']:.4f}   (higher is better)")
    print(f"    Pfa      : {res['False Alarm Rate (Pfa)']:.4f}   (lower is better)")
    print(f"    AUC-ROC  : {res['AUC-ROC']:.4f}")
    print(f"    Inference: {res['Inference Time (ms/sample)']:.4f} ms/sample")
print('\n  Primary metrics: Pd (maximise) and Pfa (minimise)')
print('='*60)